# Klasifikasi Ketebalan RNFL per sub-ROI (4 kelas) - Transfer Learning + CV per pasien
Fitur: crop wedge, kanonikalisasi mata kiri/kanan, input posisi sektor, oversampling hanya pada data train,
perbandingan skenario oversampling 0.5 vs 1.0, metrik macro-F1 / balanced accuracy / recall per kelas.

In [ ]:
import os, glob, re, shutil, zipfile, numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, recall_score)
import matplotlib.pyplot as plt

print(tf.__version__, tf.config.list_physical_devices('GPU'))   # harus ada GPU (T4)

# Deteksi platform otomatis: Kaggle atau Google Colab
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    OUT_DIR = '/kaggle/working/hasil_training/'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Penelitian_RNFL/'
    OUT_DIR  = BASE_DIR + 'hasil_training/'
os.makedirs(OUT_DIR, exist_ok=True)
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| OUT_DIR =', OUT_DIR)

In [ ]:
# Lokasi data. Kaggle: cari otomatis di /kaggle/input (Kaggle biasanya sudah mengekstrak zip yang diunggah sebagai Dataset).
if ON_KAGGLE:
    CSV_PATH = glob.glob('/kaggle/input/**/label_sector_clean.csv', recursive=True)[0]
    found = glob.glob('/kaggle/input/**/*_sector_0.png', recursive=True)
    if found:
        IMAGE_DIR = os.path.dirname(found[0]) + '/'
    else:   # dataset berisi zip yang belum terekstrak
        zp = glob.glob('/kaggle/input/**/sector_clean.zip', recursive=True)[0]
        with zipfile.ZipFile(zp) as z: z.extractall('/tmp/')
        IMAGE_DIR = '/tmp/sector/'
    # Lanjutkan eksperimen dari output versi sebelumnya (jika dilampirkan sebagai input notebook)
    for p in glob.glob('/kaggle/input/**/hasil_cv.csv', recursive=True):
        if 'hasil_training' in p and not os.path.exists(OUT_DIR + 'hasil_cv.csv'):
            src_dir = os.path.dirname(p)
            for f in glob.glob(src_dir + '/cm_*.npy') + glob.glob(src_dir + '/.rerun_*') + [p]:
                shutil.copy(f, OUT_DIR)
            print('Melanjutkan dari hasil sebelumnya:', src_dir)
else:
    CSV_PATH  = BASE_DIR + 'label_sector_clean.csv'
    ZIP_PATH  = BASE_DIR + 'sector_clean.zip'
    IMAGE_DIR = '/content/sector/'          # disk lokal Colab (jauh lebih cepat daripada dari Drive)
    if not os.path.isdir(IMAGE_DIR) or len(os.listdir(IMAGE_DIR)) < 5000:
        with zipfile.ZipFile(ZIP_PATH) as z: z.extractall('/content/')
print('CSV  :', CSV_PATH)
print('Gambar:', IMAGE_DIR, '->', len(os.listdir(IMAGE_DIR)), 'file (harus 5180)')

In [ ]:
# Inspeksi CSV
df_raw = pd.read_csv(CSV_PATH)
print(df_raw.shape); display(df_raw.head()); print(df_raw['label'].value_counts().sort_index())

In [ ]:
# ================= KONFIGURASI =================
FILE_COL, LABEL_COL = 'file_name', 'label'
GROUP_REGEX = r'^(fff_\d+_\d+)_'      # 1 pasien = mata kiri + kanan + semua scan ulang (sisi bisa l / r / u)
NAME_MAP = {1: 'kelas_1', 2: 'kelas_2', 3: 'kelas_3', 4: 'kelas_4'}   # arti kelas belum dipastikan -> ganti bila sudah jelas

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
N_FOLDS    = 5
RUN_FOLDS  = None        # None = semua fold; [0] = uji cepat 1 fold
SEED       = 42
EPOCHS_HEAD, EPOCHS_FT = 8, 12     # dikurangi agar muat di batas runtime Colab (EarlyStopping tetap aktif)
PATIENCE   = 3
USE_SECTOR_INPUT = True  # tambahkan posisi sektor kanonik sebagai input tambahan
USE_MIXED_PRECISION = False   # True = lebih cepat di GPU T4 (~1,5-2x); jika muncul error dtype, kembalikan ke False
if USE_MIXED_PRECISION: keras.mixed_precision.set_global_policy('mixed_float16')

# Urutan: dua arsitektur yang belum diuji dulu; MobileNetV2 (paling ringan) terakhir.
ARCHS = ['ResNet50', 'InceptionV3', 'MobileNetV2']
# Hasil lama arsitektur di bawah ini dibuang dan diulang dengan pengaturan yang SAMA dengan arsitektur lain (epoch, patience, dst),
# supaya perbandingan adil. Hasil lama dicadangkan ke hasil_cv_backup.csv. Kosongkan list ini untuk mempertahankan hasil lama.
RERUN_ARCHS = ['MobileNetV2']
# Class weight (skenario A) dibuang karena terbukti terburuk. Fokus oversampling train: alpha 0.5 dan 1.0.
SCENARIOS = [('B_oversample_0.5', 0.5), ('C_oversample_1.0', 1.0)]
RESULT_CSV = OUT_DIR + 'hasil_cv.csv'   # hasil lama dibaca dulu (tidak ditimpa); kombinasi yang sudah selesai dilewati otomatis
# Jika hasil_cv.csv berasal dari versi lama notebook (tanpa kolom 'scenario'), pindahkan agar tidak error/bercampur
if os.path.exists(RESULT_CSV) and 'scenario' not in pd.read_csv(RESULT_CSV, nrows=1).columns:
    os.rename(RESULT_CSV, OUT_DIR + 'hasil_cv_versi_lama.csv'); print('hasil_cv.csv lama dipindah ke hasil_cv_versi_lama.csv')

In [ ]:
df = df_raw.copy()
df['path'] = df[FILE_COL].astype(str).apply(lambda f: os.path.join(IMAGE_DIR, f))
missing = ~df['path'].apply(os.path.exists)
print('File hilang:', missing.sum(), 'dari', len(df))
df = df[~missing].reset_index(drop=True)

df['group'] = df[FILE_COL].astype(str).str.extract(GROUP_REGEX)[0]
assert df['group'].notna().all(), 'ada nama file yang tidak cocok GROUP_REGEX'
print('Jumlah pasien:', df['group'].nunique(), '| baris:', len(df))

# Sisi mata: l = kiri; r dan u = kanan
df['side'] = df[FILE_COL].astype(str).str.extract(r'^fff_\d+_\d+_([a-z])_')[0].map({'l': 'L', 'r': 'R', 'u': 'R'})
assert df['side'].notna().all(), 'ada sisi mata tak dikenal'
df['sec'] = df[FILE_COL].astype(str).str.extract(r'sector_(\d+)')[0].astype(int)
# Indeks sektor di kanvas sama untuk kiri/kanan, tetapi anatomis mata kiri = bayangan cermin mata kanan:
# sektor k (kiri) ~ sektor (10-k)%12 (kanan). Sumbu vertikal (5 dan 11) tidak berubah.
df['csec'] = np.where(df['side'] == 'L', (10 - df['sec']) % 12, df['sec'])
df['flip'] = (df['side'] == 'L').astype(int)    # wedge mata kiri di-flip horizontal agar orientasi serat sama dengan kanan
print(df['side'].value_counts())

le = LabelEncoder()
df['y'] = le.fit_transform(df[LABEL_COL])
CLASS_NAMES = [NAME_MAP.get(c, str(c)) for c in le.classes_]; NUM_CLASSES = len(CLASS_NAMES)
print(CLASS_NAMES); print(df['y'].value_counts().sort_index())
print('Pasien per kelas:', {n: df.loc[df['y'] == i, 'group'].nunique() for i, n in enumerate(CLASS_NAMES)})

In [ ]:
PREPROCESS = {
    'MobileNetV2': keras.applications.mobilenet_v2.preprocess_input,
    'ResNet50':    keras.applications.resnet50.preprocess_input,
    'InceptionV3': keras.applications.inception_v3.preprocess_input,
}
BUILDERS = {
    'MobileNetV2': keras.applications.MobileNetV2,
    'ResNet50':    keras.applications.ResNet50,
    'InceptionV3': keras.applications.InceptionV3,
}

# Augmentasi ringan. TIDAK ada flip acak (flip hanya dipakai deterministik untuk menyamakan mata kiri dengan kanan).
augment = keras.Sequential([
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.05),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
])

def crop_wedge(img):
    """Gambar asli = kanvas 448x448 hitam dengan wedge kecil. Crop ke bounding box wedge agar detail tidak hilang saat resize."""
    m = tf.reduce_max(img, axis=-1) > 12
    idx = tf.cast(tf.where(m), tf.int32)          # tf.where -> int64; samakan ke int32
    def _crop():
        y0 = tf.reduce_min(idx[:, 0]); x0 = tf.reduce_min(idx[:, 1])
        y1 = tf.reduce_max(idx[:, 0]) + 1; x1 = tf.reduce_max(idx[:, 1]) + 1
        return img[y0:y1, x0:x1, :]
    return tf.cond(tf.shape(idx)[0] > 20, _crop, lambda: img)

def make_ds(paths, labels, flips, secs, preprocess, training):
    def load(p, y, fl, sc):
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img = crop_wedge(tf.cast(img, tf.float32))
        img = tf.cond(fl > 0, lambda: tf.image.flip_left_right(img), lambda: img)   # kanonikalisasi mata kiri -> kanan
        img = tf.image.resize_with_pad(img, IMG_SIZE[0], IMG_SIZE[1])
        return img, tf.one_hot(sc, 12), y
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, flips, secs))
    if training: ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)
    def prep(x, s_, y):
        if training: x = augment(x, training=True)
        x = preprocess(x)
        return ({'img': x, 'sec': s_}, y) if USE_SECTOR_INPUT else (x, y)
    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

def build_model(arch):
    base = BUILDERS[arch](include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inp = keras.Input(IMG_SIZE + (3,), name='img')
    x = layers.GlobalAveragePooling2D()(base(inp, training=False))
    if USE_SECTOR_INPUT:
        sec_in = keras.Input((12,), name='sec')
        x = layers.Concatenate()([x, sec_in]); inputs = {'img': inp, 'sec': sec_in}
    else:
        inputs = inp
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)   # float32 agar aman dengan mixed precision
    return keras.Model(inputs, out), base

def compile_model(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

def callbacks(ckpt):
    return [keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7),
            keras.callbacks.ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True, save_weights_only=True)]

def oversample(dtr, alpha=0.5, seed=SEED):
    """Hanya untuk data TRAIN. alpha=0 -> tanpa oversampling; alpha=1 -> semua kelas disamakan dengan kelas terbesar."""
    n = dtr['y'].value_counts(); nmax = n.max(); parts = []
    for c, nc in n.items():
        target = int(round(nc * (nmax / nc) ** alpha))
        g = dtr[dtr['y'] == c]
        parts.append(g if target <= nc else pd.concat([g, g.sample(target - nc, replace=True, random_state=seed)]))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

In [ ]:
def run_cv(arch, scenario, alpha, done):
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (tr, va) in enumerate(splitter.split(df['path'], df['y'], df['group'])):
        if RUN_FOLDS is not None and fold not in RUN_FOLDS: continue
        if (scenario, arch, fold) in done:
            print(f'lewati (sudah ada): {scenario} | {arch} | fold {fold}'); continue
        dtr, dva = df.iloc[tr], df.iloc[va]
        assert not (set(dtr['group']) & set(dva['group'])), 'kebocoran: pasien sama ada di train dan validasi'

        dfit = oversample(dtr, alpha)    # oversampling hanya pada TRAIN; class weight tidak dipakai
        print(f'\n===== {scenario} | {arch} | fold {fold+1}/{N_FOLDS} | train={len(dfit)} (asli {len(dtr)}) val={len(dva)} =====')

        pre = PREPROCESS[arch]
        ds_tr = make_ds(dfit['path'].values, dfit['y'].values, dfit['flip'].values, dfit['csec'].values, pre, True)
        ds_va = make_ds(dva['path'].values, dva['y'].values, dva['flip'].values, dva['csec'].values, pre, False)  # validasi: distribusi asli

        keras.backend.clear_session()
        model, base = build_model(arch)
        ckpt = f'{OUT_DIR}{scenario}_{arch}_fold{fold}.weights.h5'

        # Fase 1: latih head saja
        compile_model(model, 1e-3)
        model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_HEAD, callbacks=callbacks(ckpt), verbose=2)
        # Fase 2: fine-tuning ~30 layer terakhir (BatchNorm tetap beku)
        base.trainable = True
        for l in base.layers[:-30]: l.trainable = False
        for l in base.layers:
            if isinstance(l, layers.BatchNormalization): l.trainable = False
        compile_model(model, 1e-5)
        model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_FT, callbacks=callbacks(ckpt), verbose=2)

        prob = model.predict(ds_va, verbose=0); pred = prob.argmax(1); yt = dva['y'].values
        try: auc = roc_auc_score(yt, prob, multi_class='ovr', average='macro', labels=np.arange(NUM_CLASSES))
        except ValueError: auc = np.nan
        rec = recall_score(yt, pred, labels=np.arange(NUM_CLASSES), average=None, zero_division=0)
        row = dict(scenario=scenario, arch=arch, fold=fold, acc=accuracy_score(yt, pred),
                   bal_acc=balanced_accuracy_score(yt, pred), macro_f1=f1_score(yt, pred, average='macro'), auc_ovr=auc)
        row.update({f'recall_{n}': r for n, r in zip(CLASS_NAMES, rec)})
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items() if k not in ('scenario', 'arch')})

        # simpan SEGERA per fold (aman jika runtime terputus; fold ini tidak diulang saat dijalankan lagi)
        prev = pd.read_csv(RESULT_CSV) if os.path.exists(RESULT_CSV) else pd.DataFrame()
        pd.concat([prev, pd.DataFrame([row])], ignore_index=True).to_csv(RESULT_CSV, index=False)
        np.save(f'{OUT_DIR}cm_{scenario}_{arch}_fold{fold}.npy', confusion_matrix(yt, pred, labels=np.arange(NUM_CLASSES)))

In [ ]:
# Jalankan. Urutan: per arsitektur dulu (kedua skenario), supaya jika waktu runtime habis perbandingan alpha 0.5 vs 1.0
# untuk satu arsitektur sudah lengkap. Jika runtime terputus, jalankan ulang sel 1-6 lalu sel ini: fold yang selesai dilewati.
prev = pd.read_csv(RESULT_CSV) if os.path.exists(RESULT_CSV) else pd.DataFrame(columns=['scenario', 'arch', 'fold'])
for ra in RERUN_ARCHS:     # dilakukan sekali saja per arsitektur (ditandai file flag) supaya aman jika runtime terputus lalu dijalankan ulang
    flag = OUT_DIR + f'.rerun_{ra}_done'
    if not os.path.exists(flag):
        if len(prev): prev.to_csv(OUT_DIR + 'hasil_cv_backup.csv', index=False)
        prev = prev[prev['arch'] != ra]
        if len(prev): prev.to_csv(RESULT_CSV, index=False)
        elif os.path.exists(RESULT_CSV): os.remove(RESULT_CSV)
        for p in glob.glob(f'{OUT_DIR}cm_*_{ra}_fold*.npy'): os.remove(p)
        open(flag, 'w').close(); print(f'hasil lama {ra} dibuang (cadangan: hasil_cv_backup.csv)')
done = set(zip(prev['scenario'], prev['arch'], prev['fold']))
for arch in ARCHS:
    for scenario, alpha in SCENARIOS:
        run_cv(arch, scenario, alpha, done)

In [ ]:
# Ringkasan: semua arsitektur yang pernah dijalankan (termasuk MobileNetV2 dari run sebelumnya), hanya skenario yang dipakai.
res = pd.read_csv(RESULT_CSV)
res = res[res['scenario'].isin([sc for sc, _ in SCENARIOS])]
metric_cols = ['acc', 'bal_acc', 'macro_f1', 'auc_ovr'] + [c for c in res.columns if c.startswith('recall_')]
summary = res.groupby(['arch', 'scenario'])[metric_cols].mean().round(4)
summary['n_fold'] = res.groupby(['arch', 'scenario']).size()
display(summary)
summary.to_csv(OUT_DIR + 'ringkasan_cv.csv')
print('Pembanding: tebakan kelas mayoritas ->', round((df['y'] == df['y'].value_counts().idxmax()).mean(), 3),
      'akurasi, balanced acc =', round(1 / NUM_CLASSES, 3))

import glob, re
combos = sorted({tuple(re.match(r'cm_(.+?)_(ResNet50|InceptionV3|MobileNetV2)_fold\d+\.npy', os.path.basename(p)).groups())
                 for p in glob.glob(OUT_DIR + 'cm_*.npy')})
combos = [c for c in combos if c[0] in [sc for sc, _ in SCENARIOS]]
if combos:
    fig, axes = plt.subplots(1, len(combos), figsize=(4.5 * len(combos), 4), squeeze=False)
    for ax, (sc, ar) in zip(axes[0], combos):
        cm = sum(np.load(p) for p in glob.glob(f'{OUT_DIR}cm_{sc}_{ar}_fold*.npy'))
        ax.imshow(cm, cmap='Blues'); ax.set_title(f'{ar}\n{sc}', fontsize=9)
        ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
        ax.set_xticklabels(CLASS_NAMES, rotation=45); ax.set_yticklabels(CLASS_NAMES)
        ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES): ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR + 'confusion_matrix.png', dpi=150); plt.show()

## Cara membaca hasil
- Bandingkan skenario B dan C dengan **macro_f1, bal_acc, dan recall per kelas**, bukan akurasi (kelas mayoritas = ~68%).
- Hasil MobileNetV2 dari run sebelumnya ikut tampil di ringkasan bila file hasil_cv.csv masih ada.
- Jika semua skenario lemah: masalahnya sinyal dan jumlah pasien, bukan teknik penyeimbangan -> pertimbangkan tugas yang lebih sederhana atau data tambahan.